[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedorShind/EMRG/blob/main/docs/tutorials/vqe_h2_mitigation.ipynb)

# VQE with EMRG

This notebook builds a small VQE-style circuit, asks EMRG what mitigation recipe it would use, and looks at the generated Mitiq code.

This is not a chemistry benchmark. There is no H2 Hamiltonian or energy objective here. The point is the workflow: inspect, choose, render.

## Install

Colab needs the preview extra for the optional simulator cell. In a local checkout, use the project virtual environment instead.

In [ ]:
!pip install -q "emrg[preview]"

## Imports

In [ ]:
from math import isinf, pi

from qiskit import QuantumCircuit

from emrg import analyze_circuit, generate_recipe

## Build a circuit

This is a deterministic 4-qubit hardware-efficient ansatz. It has non-Clifford rotations and a short chain of entangling gates.

In [ ]:
qc = QuantumCircuit(4, 4)

angles = [
    [0.31 * pi, 0.17 * pi, 0.43 * pi, 0.29 * pi],
    [0.37 * pi, 0.11 * pi, 0.23 * pi, 0.41 * pi],
]

for layer_angles in angles:
    for qubit, theta in enumerate(layer_angles):
        qc.ry(theta, qubit)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.cx(2, 3)

qc.measure(range(4), range(4))
qc.draw("text")

## Analyze the circuit

EMRG starts by reading circuit features. No simulation is needed for this step.

In [ ]:
features = analyze_circuit(qc)

feature_summary = {
    "qubits": features.num_qubits,
    "depth": features.depth,
    "total gates": features.total_gate_count,
    "multi-qubit gates": features.multi_qubit_gate_count,
    "noise category": features.noise_category,
    "non-Clifford gates": features.non_clifford_count,
    "non-Clifford fraction": f"{features.non_clifford_fraction:.1%}",
    "PEC overhead estimate": f"{features.pec_overhead_estimate:.2f}x",
}

for name, value in feature_summary.items():
    print(f"{name:24} {value}")

## Ask EMRG for a recipe

The default policy considers CDR for circuits above 30% non-Clifford gates and depth 12 through 40. This ansatz has enough non-Clifford gates, but it is depth 8, so EMRG stays with ZNE.

In [ ]:
def print_recipe(recipe):
    print(f"Technique:          {recipe.technique}")
    print(f"Factory:            {recipe.factory_name or 'none'}")
    print(f"Scale factors:      {recipe.scale_factors or 'none'}")
    print(f"Scaling method:     {recipe.scaling_method or 'none'}")
    print(f"Estimated overhead: {recipe.estimated_overhead:.1f}x")
    if recipe.factory_kwargs:
        print(f"Factory kwargs:     {dict(recipe.factory_kwargs)}")
    if recipe.warnings:
        print("Warnings:")
        for warning in recipe.warnings:
            print(f"  - {warning}")
    else:
        print("Warnings:           none")


result = generate_recipe(qc)
print_recipe(result.recipe)

## Look at the generated code

EMRG renders Mitiq-native code. The generated code still needs your backend executor, so the notebook prints only the first useful part.

In [ ]:
def show_code_excerpt(code, max_lines=45):
    lines = code.splitlines()
    for line in lines[:max_lines]:
        print(line)
    if len(lines) > max_lines:
        print(f"... {len(lines) - max_lines} more lines")


show_code_excerpt(result.code)

## Optional preview

This circuit is small enough for the local preview path. The numbers below are simulator feedback, not hardware evidence.

In [ ]:
preview_result = generate_recipe(qc, preview=True, noise_level=0.01)
p = preview_result.preview

if p is None:
    print("Preview skipped: preview dependencies are not installed")
elif p.ideal_value is None:
    print(f"Preview skipped: {p.warning}")
else:
    reduction = "infinite" if isinf(p.error_reduction) else f"{p.error_reduction:.1f}x"
    print(f"Technique:  {p.technique}")
    print(f"Ideal:      {p.ideal_value:+.4f}")
    print(f"Noisy:      {p.noisy_value:+.4f}  (error: {p.noisy_error:.4f})")
    print(f"Mitigated:  {p.mitigated_value:+.4f}  (error: {p.mitigated_error:.4f})")
    print(f"Reduction:  {reduction}")
    if p.warning:
        print(f"Note:       {p.warning}")

## Optional override

Forcing a technique is useful for comparison, but it bypasses the automatic choice. EMRG keeps warnings on the recipe when the forced technique falls outside the policy window.

In [ ]:
forced = generate_recipe(qc, technique="cdr").recipe
print_recipe(forced)

## Summary

EMRG is a recipe layer. For this short VQE-style ansatz, it inspects the circuit and renders a ZNE recipe with global folding.

Policies are data, not code. In a project, tune the policy file when the default thresholds or Mitiq settings are not the right fit.